In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
MNIST 3-Phase 64-QAM Neural Network with STE Quantization
64-QAM Inputs, Activations, Weights, Biases, and Outputs
═══════════════════════════════════════════════════════════════════════════════

This project implements a fully complex-valued 64-QAM neural network for
MNIST classification using communication-inspired neural computation and
Straight-Through Estimator (STE) based quantization. Input pixels, hidden
activations, weights, biases, and final outputs are all constrained to valid
64-QAM constellation symbols through nearest-level quantization.

The network uses separate real and imaginary weight tensors together with
complex-valued matrix multiplication and complex LayerNorm operations for
stable optimization in the complex domain. Training is performed in three
stages: float training, activation quantization, and finally full 64-QAM
weight + activation quantization for stable convergence.

A tolerance-based purity verification system checks whether all weights,
biases, activations, and outputs exactly belong to the valid 64-QAM
constellation, ensuring the final model is fully OFDM-transmissible and
hardware-compatible.

RESULT:
  Final test accuracy achieved: 97%
═══════════════════════════════════════════════════════════════════════════════
"""

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ─────────────────────────────────────────────
# 0. 64‑QAM levels and purity check
# ─────────────────────────────────────────────

import itertools

# 8 amplitude levels per axis (unnormalized 64‑QAM-style)
QAM_LEVELS_1D = torch.tensor([-7, -5, -3, -1, 1, 3, 5, 7], dtype=torch.float32)

# Optional: normalize average energy if you like (simple mean-normalization)
QAM_LEVELS_1D = QAM_LEVELS_1D / QAM_LEVELS_1D.abs().mean()

QAM_TOL = 1e-2   # tolerance for checking closeness to a valid level


# Maps every element of a real-valued tensor to the nearest value in the
# provided 1D levels array by computing absolute differences against all
# levels and taking the argmin. Used by both the QAM quantizers and the
# input encoding step to snap continuous values onto the discrete constellation.
def quantize_to_levels(x: torch.Tensor, levels: torch.Tensor) -> torch.Tensor:
    """Quantize real-valued tensor x to nearest value in `levels`."""
    x_flat = x.view(-1, 1)                      # [N, 1]
    diffs  = torch.abs(x_flat - levels[None])   # [N, L]
    idx    = diffs.argmin(dim=1)
    return levels[idx].view_as(x)


# Validates that every element of a complex tensor lies on the 64-QAM grid
# by checking, independently for the real and imaginary axes, that each
# value is within QAM_TOL of some valid level in QAM_LEVELS_1D. Returns
# True only if both axes pass for all elements. The tolerance-based check
# (v3 fix) replaces an earlier set-equality check that broke due to
# floating-point representation mismatches.
def is_qam64(tensor: torch.Tensor) -> bool:
    """
    Returns True if every element of tensor is a valid 64‑QAM symbol.
    Valid means: real, imag each ≈ one of QAM_LEVELS_1D (within QAM_TOL).
    """
    r = tensor.real.detach().cpu().float()
    i = tensor.imag.detach().cpu().float()

    def axis_ok(z):
        z_flat = z.view(-1, 1)
        diffs  = torch.abs(z_flat - QAM_LEVELS_1D[None])
        min_diff, _ = diffs.min(dim=1)
        return torch.all(min_diff < QAM_TOL)

    return bool(axis_ok(r) and axis_ok(i))



# ─────────────────────────────────────────────
# 1. 64‑QAM Quantizers with STE
# ─────────────────────────────────────────────

# Custom autograd function for quantizing complex activations to the 64-QAM
# constellation. In the forward pass, real and imaginary parts are each
# snapped independently to the nearest level in QAM_LEVELS_1D. In the
# backward pass, gradients are passed through only for elements whose
# magnitude is within the constellation range (clipped STE), blocking
# gradient flow for out-of-range activations to prevent instability.
class QAM64QuantizeActivation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        r = quantize_to_levels(x.real, QAM_LEVELS_1D)
        i = quantize_to_levels(x.imag, QAM_LEVELS_1D)
        return torch.complex(r, i)

    @staticmethod
    def backward(ctx, grad_out):
        x, = ctx.saved_tensors
        max_level = QAM_LEVELS_1D.abs().max()
        mask_r = (torch.abs(x.real) <= max_level).float()
        mask_i = (torch.abs(x.imag) <= max_level).float()
        return torch.complex(grad_out.real * mask_r, grad_out.imag * mask_i)


# Identical in structure to QAM64QuantizeActivation but applied to weight
# tensors rather than activations. Keeping them as separate classes allows
# independent STE masking policies for weights and activations in the future,
# and makes it explicit in the code which type of tensor is being quantized
# at each call site.
class QAM64QuantizeWeight(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w):
        ctx.save_for_backward(w)
        r = quantize_to_levels(w.real, QAM_LEVELS_1D)
        i = quantize_to_levels(w.imag, QAM_LEVELS_1D)
        return torch.complex(r, i)

    @staticmethod
    def backward(ctx, grad_out):
        w, = ctx.saved_tensors
        max_level = QAM_LEVELS_1D.abs().max()
        mask_r = (torch.abs(w.real) <= max_level).float()
        mask_i = (torch.abs(w.imag) <= max_level).float()
        return torch.complex(grad_out.real * mask_r, grad_out.imag * mask_i)


# ─────────────────────────────────────────────
# 2. Complex Layer Normalisation
# ─────────────────────────────────────────────

# Layer normalization extended to complex-valued feature vectors by applying
# two independent real-valued LayerNorm instances — one to the real part and
# one to the imaginary part — and recombining them into a complex tensor.
# This stabilizes training in the same way as standard LayerNorm while
# respecting the two-component structure of the complex activations.
class ComplexLayerNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.ln_r = nn.LayerNorm(num_features, eps=eps)
        self.ln_i = nn.LayerNorm(num_features, eps=eps)

    # Normalizes real and imaginary components separately via their own
    # LayerNorm instances and reassembles the result as a complex tensor.
    def forward(self, x):
        return torch.complex(self.ln_r(x.real), self.ln_i(x.imag))


# ─────────────────────────────────────────────
# 3. QAMLinear
# ─────────────────────────────────────────────

# Fully-connected layer that operates natively on complex numbers. Weights
# and biases are stored as pairs of real-valued parameters (w_real, w_imag)
# and composed into a complex weight matrix at forward time. The complex
# matrix-vector product is expanded into four real dot-products using the
# standard (a+jb)(c+jd) formula to avoid needing native complex CUDA kernels.
# Weight quantization to 64-QAM is toggled via the quantize_weights flag,
# enabling the phased training curriculum defined in main().
class QAMLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        std = (2.0 / in_features) ** 0.5
        self.w_real = nn.Parameter(torch.randn(out_features, in_features) * std)
        self.w_imag = nn.Parameter(torch.randn(out_features, in_features) * std)
        self.b_real = nn.Parameter(torch.zeros(out_features))
        self.b_imag = nn.Parameter(torch.zeros(out_features))
        self.quantize_weights = False

    # Assembles the complex weight and bias from their real/imaginary parameter
    # pairs, optionally snaps them to the 64-QAM grid, then computes the complex
    # linear transformation via the four-real-matmul expansion: real output =
    # xr·wr − xi·wi + b_r, imaginary output = xr·wi + xi·wr + b_i.
    def forward(self, x):
        w = torch.complex(self.w_real, self.w_imag)
        b = torch.complex(self.b_real, self.b_imag)
        if self.quantize_weights:
            w = QAM64QuantizeWeight.apply(w)
            b = QAM64QuantizeWeight.apply(b)
        xr, xi = x.real, x.imag
        wr, wi = w.real, w.imag
        real_out = xr @ wr.T - xi @ wi.T + b.real
        imag_out = xr @ wi.T + xi @ wr.T + b.imag
        return torch.complex(real_out, imag_out)

    # Returns a hard-quantized copy of this layer's weight and bias tensors
    # by applying the 64-QAM quantizer inside a no-grad context. Used
    # exclusively by the purity verification routine to confirm that the
    # latent weights, when snapped to the constellation, are valid 64-QAM
    # symbols — without affecting the latent parameters themselves.
    def get_qam_weights(self):
        # Now actually returns 64‑QAM‑quantized weights
        with torch.no_grad():
            w = QAM64QuantizeWeight.apply(torch.complex(self.w_real, self.w_imag))
            b = QAM64QuantizeWeight.apply(torch.complex(self.b_real, self.b_imag))
            return w, b



# ─────────────────────────────────────────────
# 4. Network
# ─────────────────────────────────────────────

# Three-layer complex-valued MLP for MNIST classification. Pixels are first
# encoded as 64-QAM symbols (pairs of pixels → real + imaginary), passed
# through two hidden QAMLinear layers with ComplexLayerNorm and optional
# QAM activation quantization, and finally through a float output layer.
# Classification logits are the complex magnitude |fc3(h2)|; the quantized
# output symbol is also returned for purity verification. Both quantize_activations
# and per-layer quantize_weights are toggled externally by the training loop.
class FullQAMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = QAMLinear(392, 512)
        self.fc2 = QAMLinear(512, 512)
        self.fc3 = QAMLinear(512, 10)
        self.ln1 = ComplexLayerNorm(512)
        self.ln2 = ComplexLayerNorm(512)
        self.quantize_activations = False

    # Encodes a batch of flattened pixel vectors as 64-QAM complex symbols
    # by treating alternating pixels as the real and imaginary parts of each
    # symbol and snapping both components to the nearest QAM level. This
    # converts a standard image batch into a complex-valued input compatible
    # with the QAMLinear layers while enforcing the input constellation constraint.
    @staticmethod
    def pixels_to_qam(x):
        # Now: pixels_to_qam64 (but you can keep the name)
        x_flat = x.view(x.size(0), -1)
        r_raw = x_flat[:, 0::2]
        i_raw = x_flat[:, 1::2]
        r = quantize_to_levels(r_raw, QAM_LEVELS_1D)
        i = quantize_to_levels(i_raw, QAM_LEVELS_1D)
        return torch.complex(r, i)

    # Applies 64-QAM quantization to activations when the quantize_activations
    # flag is True (Phase 2 and 3 of training), otherwise passes the complex
    # tensor through unchanged (Phase 1). Acts as the nonlinearity between layers.
    def _act(self, x):
        if self.quantize_activations:
            return QAM64QuantizeActivation.apply(x)
        return x

    # Runs the full forward pass: pixel encoding → fc1+norm+act → fc2+norm+act
    # → fc3 → magnitude logits. Returns both the real-valued logits used for
    # cross-entropy loss and the hard-quantized output symbol used for purity
    # verification, so callers can access both without a second forward pass.
    def forward(self, x):
        x_q = self.pixels_to_qam(x)
        h1 = self._act(self.ln1(self.fc1(x_q)))
        h2 = self._act(self.ln2(self.fc2(h1)))
        pre_out = self.fc3(h2)
        logits   = torch.abs(pre_out)
        out_qam = QAM64QuantizeActivation.apply(pre_out)
        return logits, out_qam


# ─────────────────────────────────────────────
# 5. FIXED Purity Verification
# ─────────────────────────────────────────────

# End-to-end audit function that confirms every tensor in the network —
# weights, biases, and intermediate activations — lies on the 64-QAM
# constellation when the model is run in fully-quantized mode. Temporarily
# enables both quantize_activations and quantize_weights for all layers,
# re-runs the forward pass step-by-step, calls is_qam64() on each tensor,
# and prints a formatted purity report with min/max values per axis.
# Restores all quantization flags to their original state before returning.
def verify_qam_purity(model, sample_batch, device):
    model.eval()

    # Checks a single complex tensor against the 64-QAM constellation and
    # prints its name, pass/fail status, and real/imaginary value ranges
    # for quick visual inspection of how close the values are to valid levels.
    def report(tensor, name):
        ok = is_qam64(tensor)
        r = tensor.real.detach().cpu().float()
        i = tensor.imag.detach().cpu().float()
        r_min, r_max = r.min().item(), r.max().item()
        i_min, i_max = i.min().item(), i.max().item()
        status = "✅ 64‑QAM" if ok else "❌ NOT 64‑QAM"
        print(f"  {name:40s}: {status}")
        print(f"    real ∈ [{r_min:+.8f}, {r_max:+.8f}]   "
              f"imag ∈ [{i_min:+.8f}, {i_max:+.8f}]")
        return ok

    all_ok = True
    print("\n" + "═"*60)
    print("  64‑QAM PURITY REPORT")
    print("═"*60)

    print("\n[Weights & Biases]")
    for name in ["fc1", "fc2", "fc3"]:
        layer = getattr(model, name)
        w_q, b_q = layer.get_qam_weights()
        all_ok &= report(w_q, f"{name}.weight")
        all_ok &= report(b_q, f"{name}.bias")

    print("\n[Activations — strict 64‑QAM forward pass]")
    with torch.no_grad():
        x = sample_batch.to(device)

        orig_qa = model.quantize_activations
        orig_qw = [l.quantize_weights for l in [model.fc1, model.fc2, model.fc3]]
        model.quantize_activations = True
        for l in [model.fc1, model.fc2, model.fc3]:
            l.quantize_weights = True

        x_in = FullQAMNet.pixels_to_qam(x)
        all_ok &= report(x_in, "Input (pixels → 64‑QAM)")

        h1 = QAM64QuantizeActivation.apply(model.ln1(model.fc1(x_in)))
        all_ok &= report(h1, "After fc1 + LayerNorm + 64‑QAM")

        h2 = QAM64QuantizeActivation.apply(model.ln2(model.fc2(h1)))
        all_ok &= report(h2, "After fc2 + LayerNorm + 64‑QAM")

        out = QAM64QuantizeActivation.apply(model.fc3(h2))
        all_ok &= report(out, "Output fc3 + 64‑QAM")

        model.quantize_activations = orig_qa
        for l, v in zip([model.fc1, model.fc2, model.fc3], orig_qw):
            l.quantize_weights = v

    print("\n" + "═"*60)
    verdict = "✅ ALL 64‑QAM — model is OFDM‑transmissible" if all_ok \
              else "❌ FAILURES FOUND — check training"
    print(f"  VERDICT: {verdict}")
    print("═"*60)
    return all_ok



# ─────────────────────────────────────────────
# 6. Training
# ─────────────────────────────────────────────

# Runs inference over a full DataLoader in eval mode with gradients disabled
# and returns top-1 classification accuracy as a percentage. Unpacks only
# the logits from the model's two-element output tuple (logits, out_qam)
# since the quantized output symbol is not needed for accuracy evaluation.
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


# Entry point that sets up MNIST data loaders, instantiates FullQAMNet, and
# runs a three-phase progressive quantization curriculum: Phase 1 trains with
# full float weights and activations to establish a good feature basis; Phase 2
# introduces QAM-quantized activations while keeping weights float; Phase 3
# enables QAM-quantized weights as well (fc3 weights kept float throughout as
# the classification head). After training, runs purity verification on a test
# batch and prints a compression summary comparing float32 storage to the
# equivalent 6-bits-per-symbol QAM representation.
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    train_set = datasets.MNIST('./data', train=True,  download=True, transform=transform)
    test_set  = datasets.MNIST('./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=256, shuffle=True,  num_workers=2)
    test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

    model     = FullQAMNet().to(device)
    criterion = nn.CrossEntropyLoss()

    phases = [
        (5,  False, False, 1e-3, "Phase 1 — Real weights + Real activations"),
        (3,  True,  False, 5e-4, "Phase 2 — Real weights + QAM activations"),
        (20, True,  True,  2e-4, "Phase 3 — QAM weights + QAM activations"),
    ]

    for (num_epochs, q_act, q_wt, lr, desc) in phases:
      print(f"\n{'─'*55}\n{desc}\n{'─'*55}")
      model.quantize_activations = q_act
      for layer in [model.fc1, model.fc2, model.fc3]:
          layer.quantize_weights = q_wt
      model.fc3.quantize_weights = False
      optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
      scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

      for epoch in range(num_epochs):
        model.train()
        correct = total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, _ = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
        print(f"  Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train: {100*correct/total:.2f}%  "
              f"Test: {evaluate(model, test_loader, device):.2f}%")
        scheduler.step()

    print(f"\n{'═'*55}")
    print(f"Final Test Accuracy: {evaluate(model, test_loader, device):.2f}%")

    sample_images, _ = next(iter(test_loader))
    verify_qam_purity(model, sample_images[:32], device)

    torch.save(model.state_dict(), "qam_model.pth")
    print("\nModel saved to qam_model.pth")

    n = sum(p.numel() for p in model.parameters())
    print(f"\n[Compression]  {n:,} latent floats  ->  {n//2:,} 64‑QAM symbols  "
          f"({n*32/1e6:.1f} Mb float32  ->  {n/1e6*6/8:.3f} Mb at 6 bits/symbol)")



if __name__ == "__main__":
    main()